# ⚽ TouchlineLM: Google Colab GPU Training Notebook (Phase 4B.1)

Welcome to the official pre-training notebook for **TouchlineLM** (`TouchlineNanoGPT`), a domain-adapted decoder-only Transformer engineered for football intelligence, tactical pattern recognition, and advanced event-level analytics (xG, xA, progressive actions, spatial positioning).

### Architectural Invariants (Audited at Commit `d345024`):
- **Model Parameters**: 4,198,912 unique trainable parameters (~4.20M)
- **Vocabulary Size**: 4,096 tokens (Custom Byte-Level BPE, zero out-of-vocabulary)
- **Max Context Length**: 1,024 tokens
- **Positional Encoding**: Rotary Position Embeddings (RoPE)
- **Normalization & Attention**: Pre-LayerNorm, PyTorch Scaled Dot-Product Attention (SDPA) with causal triangular masking
- **Weight Tying**: Output language model head tied to input token embeddings
- **Corpus**: FIFA World Cup 2022 StatsBomb Open Data (509,677 train tokens, 88,651 val tokens)
- **Training Budget**: 3,000 steps with $B=16, T=512 \implies 24,576,000$ tokens seen (~48.2 epochs over train split)

---

### Notebook Structure:
1. **Environment Setup & Verification**: Hardware checks (CUDA fail-fast), git clone at audited commit (`d345024`), dependency installation, Google Drive mounting.
2. **Dataset & Model Verification**: Tokenizer validation, binary token cache inspection, parameter count audit, GPU forward pass.
3. **GPU Smoke Test**: Conservative 15-step test in isolated `smoke_test/` directory validating CUDA VRAM allocation, loss convergence, atomic checkpoint saving (`smoke_test_model.pt`), reload, and 5-step continuation.
4. **Full Training Configuration**: Hyperparameter control center with token budget analysis and non-auto-triggered CLI command execution.
5. **Resumption & Interactive Inference**: Resuming training after disconnects and testing generation on tactical prompts.

## 1. Environment Setup & Hardware Verification

Run the cells in this section to verify Python/PyTorch versions, confirm that Google Colab assigned an active NVIDIA GPU, install dependencies, and mount your Google Drive for persistent checkpoint storage.

In [ ]:
import sys
import torch

print("=" * 65)
print("SYSTEM & HARDWARE VERIFICATION")
print("=" * 65)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

# Fail-fast assertion: Never silently fall back to CPU for GPU training
if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL ERROR: CUDA GPU is NOT detected! Never run TouchlineLM training on CPU.\n"
        "Please enable GPU acceleration in Google Colab:\n"
        "  1. Go to top menu: Runtime -> Change runtime type\n"
        "  2. Hardware accelerator: Select 'T4 GPU' (or A100/V100 if Colab Pro)\n"
        "  3. Click 'Save' and rerun this cell."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
total_vram_gb = device_props.total_memory / (1024 ** 3)

print(f"Detected GPU:    {device_name}")
print(f"Compute Cap.:    {device_props.major}.{device_props.minor}")
print(f"Dedicated VRAM:  {total_vram_gb:.2f} GB")
print("=" * 65)
print("GPU Verification: PASSED (CUDA Ready)")

In [ ]:
import os
from pathlib import Path

# Target audited commit for Phase 4B / Phase 4B.1 reproducibility
# Commit 'd345024' contains audited training pipeline with --grad-accum-steps
TARGET_COMMIT = "d345024"

# Clone TouchlineLM repository if not already present
if not Path("src/model").exists() and not Path("TouchlineLM").exists():
    print(f"Cloning TouchlineLM repository from GitHub and checking out audited commit {TARGET_COMMIT}...")
    !git clone https://github.com/Athullvr/TouchlineLM.git TouchlineLM
    %cd TouchlineLM
    !git checkout {TARGET_COMMIT}
elif Path("TouchlineLM").exists() and not Path("src/model").exists():
    %cd TouchlineLM
    !git checkout {TARGET_COMMIT}
else:
    # Already in repository directory: ensure on correct commit
    print(f"Ensuring repository is checked out at audited commit {TARGET_COMMIT}...")
    !git checkout {TARGET_COMMIT}

# Ensure we are on the audited commit and check git status
print("\nRepository Version Verification:")
!git log -n 1 --oneline
print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
# Install required dependencies
print("Installing project dependencies from requirements.txt...")
!pip install -q -r requirements.txt
print("Dependencies installation complete.")

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive for permanent checkpoint persistence
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Configure persistent checkpoint directory on Google Drive
GDRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/TouchlineLM_Checkpoints')
GDRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print(f"Persistent Checkpoint Directory: {GDRIVE_CHECKPOINT_DIR}")
saved_checkpoints = list(GDRIVE_CHECKPOINT_DIR.glob('*.pt'))
print(f"Existing Checkpoints in Drive:   {len(saved_checkpoints)} found")
for ckpt in saved_checkpoints:
    print(f"  • {ckpt.name} ({ckpt.stat().st_size / (1024**2):.1f} MB)")
print("=" * 65)

## 2. Dataset & Model Verification

Verify that the custom Byte-Level BPE tokenizer, model configuration, tokenized binary cache, and parameter dimensions are 100% aligned with audited Phase 4A / Phase 4B specifications before executing any training.

In [ ]:
import json
import torch
from src.tokenizer.bpe_tokenizer import TouchlineBPETokenizer
from src.model.config import ModelConfig
from src.model.model import TouchlineNanoGPT

print("=" * 65)
print("1. LOADING TOKENIZER & MODEL CONFIGURATION")
print("=" * 65)

# 1. Load Tokenizer
tokenizer = TouchlineBPETokenizer.load("configs/tokenizer")
print(f"Tokenizer Loaded: Vocab Size = {tokenizer.vocab_size} (Target: 4,096)")
assert tokenizer.vocab_size == 4096, f"Expected 4096 vocab, got {tokenizer.vocab_size}"

# Test special token encoding
test_text = "<|narrative|> # Match Report: Argentina vs France <|endoftext|>"
test_tokens = tokenizer.encode(test_text, allowed_special="all")
assert 4092 in test_tokens, "Special token <|endoftext|> not recognized!"
print(f"Sample Encoding: '{test_text}' -> {test_tokens}")

# 2. Load Model Config
model_config = ModelConfig.from_json("configs/model_config.json")
print("\nModel Configuration (configs/model_config.json):")
print(json.dumps(model_config.to_dict(), indent=2))
print("=" * 65)

In [ ]:
import numpy as np
from pathlib import Path
from src.training.dataset import prepare_tokenized_dataset

print("=" * 65)
print("2. VERIFYING TOKENIZED BINARY DATASETS")
print("=" * 65)

cache_dir = Path("data/intermediate/token_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
train_bin = cache_dir / "train.bin"
val_bin = cache_dir / "val.bin"

# Prepare or load cached binary token arrays (uint16)
if not train_bin.exists():
    print("Tokenizing train split (data/processed/splits/train.txt) -> train.bin...")
    train_tokens = prepare_tokenized_dataset("data/processed/splits/train.txt", tokenizer, bin_path=train_bin)
else:
    print("Loading cached training tokens from memory-mapped train.bin...")
    train_tokens = np.memmap(train_bin, dtype=np.uint16, mode="r")

if not val_bin.exists():
    print("Tokenizing validation split (data/processed/splits/val.txt) -> val.bin...")
    val_tokens = prepare_tokenized_dataset("data/processed/splits/val.txt", tokenizer, bin_path=val_bin)
else:
    print("Loading cached validation tokens from memory-mapped val.bin...")
    val_tokens = np.memmap(val_bin, dtype=np.uint16, mode="r")

print(f"Train Tokens Count: {len(train_tokens):,} (Expected audited count: 509,677)")
print(f"Val Tokens Count:   {len(val_tokens):,} (Expected audited count: 88,651)")
print(f"Train Min Token ID: {train_tokens.min()} (Must be >= 0)")
print(f"Train Max Token ID: {train_tokens.max()} (Must be < 4096)")

# Invariant checks
assert len(train_tokens) == 509677, f"Unexpected train tokens count: {len(train_tokens)}"
assert len(val_tokens) == 88651, f"Unexpected val tokens count: {len(val_tokens)}"
assert train_tokens.min() >= 0 and train_tokens.max() < 4096
assert val_tokens.min() >= 0 and val_tokens.max() < 4096
print("Dataset Verification: 100% PASSED")
print("=" * 65)

In [ ]:
print("=" * 65)
print("3. MODEL INSTANTIATION & GPU FORWARD PASS")
print("=" * 65)

# Instantiate TouchlineNanoGPT directly on CUDA
model = TouchlineNanoGPT(model_config).to("cuda")

total_params = model.get_num_params(non_embedding=False)
non_emb_params = model.get_num_params(non_embedding=True)

print(f"Total Unique Trainable Parameters: {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"Non-Embedding Parameters:          {non_emb_params:,} (~{non_emb_params/1e6:.2f}M)")

# Assert audited exact parameter breakdown
assert total_params == 4198912, f"Parameter count mismatch: {total_params} != 4,198,912"
assert non_emb_params == 3150336, f"Non-embedding mismatch: {non_emb_params} != 3,150,336"

# Run a test forward pass with next-token cross-entropy loss
test_b, test_t = 2, 64
dummy_input = torch.randint(0, 4096, (test_b, test_t), device="cuda")
dummy_targets = torch.randint(0, 4096, (test_b, test_t), device="cuda")

model.eval()
with torch.no_grad():
    test_logits, test_loss = model(dummy_input, targets=dummy_targets)

print(f"Forward Pass Output Logits Shape:  {list(test_logits.shape)} (Expected: [2, 64, 4096])")
print(f"Forward Pass Cross-Entropy Loss:   {test_loss.item():.4f} (Finite: {torch.isfinite(test_loss).item()})")

assert test_logits.shape == (test_b, test_t, 4096)
assert torch.isfinite(test_loss).item()
print("GPU Forward Pass Verification: 100% PASSED")
print("=" * 65)

## 3. GPU Smoke Test (Separate & Non-Destructive)

This section executes a **conservative 15-step GPU smoke test**. It uses small batch dimensions (`batch_size=4, seq_len=256`) to verify:
1. **Real GPU Execution**: Confirms active CUDA kernel launches and VRAM allocation.
2. **Convergence & Loss Computation**: Verifies finite cross-entropy loss and validation perplexity.
3. **Isolated Checkpoint Persistence**: Saves exclusively to `GDRIVE_CHECKPOINT_DIR / "smoke_test" / "smoke_test_model.pt"`.
4. **State Restoration & Continuation**: Restores model weights, optimizer momentum, cosine learning rate schedule, and RNG state to execute 5 continuation steps without warmup restart.

> 🛡️ **Checkpoint Isolation Guarantee**: Smoke test artifacts are strictly isolated in the `smoke_test/` subfolder. They **cannot** overwrite, collide with, or be confused for full-training checkpoints (`best_model.pt`, `checkpoint_step_500.pt`, `final_model.pt`) in the root checkpoint directory.
>
> ⚠️ **Note**: This smoke test terminates automatically after 15 steps. It does **NOT** initiate the full training run.

In [ ]:
from src.training.trainer import TouchlineTrainer, TrainingConfig

# Isolated smoke test directory on Google Drive
smoke_dir = GDRIVE_CHECKPOINT_DIR / "smoke_test"
smoke_dir.mkdir(parents=True, exist_ok=True)

# Conservative smoke test configuration
smoke_config = TrainingConfig(
    batch_size=4,
    seq_len=256,
    grad_accum_steps=1,
    learning_rate=6e-4,
    min_lr=6e-5,
    max_steps=15,
    warmup_steps=5,
    eval_interval=5,
    eval_iters=5,
    save_interval=5,
    log_interval=1,
    device="cuda",
    checkpoint_dir=str(smoke_dir),
    seed=42,
)

print("=" * 65)
print("EXECUTING 15-STEP GPU SMOKE TEST")
print("=" * 65)

# Reset peak memory stats
torch.cuda.reset_peak_memory_stats()
initial_vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)
print(f"Initial GPU VRAM Allocated: {initial_vram_mb:.2f} MB")

smoke_model = TouchlineNanoGPT(model_config).to("cuda")
smoke_trainer = TouchlineTrainer(
    model=smoke_model,
    config=smoke_config,
    train_data=train_tokens,
    val_data=val_tokens,
    tokenizer=tokenizer,
)

# Run smoke test training loop
smoke_results = smoke_trainer.train()

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
print("=" * 65)
print(f"Smoke Test Completed Successfully at Step: {smoke_results['final_step']}")
print(f"Peak GPU VRAM Allocated During Steps:      {peak_vram_mb:.2f} MB")
print(f"Best Validation Loss Achieved:             {smoke_results['best_val_loss']:.4f}")

# Save explicit isolated smoke test checkpoint
smoke_ckpt_path = smoke_dir / "smoke_test_model.pt"
smoke_trainer.save_checkpoint(smoke_ckpt_path, val_loss=smoke_results['best_val_loss'])

assert smoke_ckpt_path.exists(), f"Checkpoint not found at {smoke_ckpt_path}"
print(f"Verified Isolated Checkpoint on Google Drive: {smoke_ckpt_path}")
print("=" * 65)

In [ ]:
print("=" * 65)
print("TESTING CHECKPOINT RELOAD & STEP CONTINUATION")
print("=" * 65)

# Instantiate a fresh model to guarantee clean weight restoration
reloaded_model = TouchlineNanoGPT(model_config).to("cuda")
reloaded_trainer = TouchlineTrainer(
    model=reloaded_model,
    config=smoke_config,
    train_data=train_tokens,
    val_data=val_tokens,
    tokenizer=tokenizer,
)

# Restore state from Google Drive isolated smoke checkpoint
restored_step = reloaded_trainer.load_checkpoint(smoke_ckpt_path)
print(f"Restored Training Step from Checkpoint: {restored_step} (Expected: {smoke_trainer.step})")
assert restored_step == smoke_trainer.step, f"Restored step mismatch: {restored_step} != {smoke_trainer.step}"

# Verify continuation: Execute 5 additional steps (Steps 16 to 20)
print("\nExecuting 5 Continuation Steps (Steps 16 to 20):")
for extra_step in range(1, 6):
    cont_loss = reloaded_trainer.train_step()
    print(f"  Step {reloaded_trainer.step:2d}/20 | Loss: {cont_loss:.4f} | LR: {reloaded_trainer.last_step_lr:.2e}")

assert reloaded_trainer.step == 20, f"Expected step 20 after continuation, got {reloaded_trainer.step}"
print("=" * 65)
print("GPU SMOKE TEST & RESUMPTION: 100% VERIFIED!")
print("=" * 65)

## 4. Full Training Configuration & Execution

This section controls the eventual **full pre-training run** on Google Colab GPU resources.

### Mathematical Token Budget Analysis:
- **Batch Size ($B$)**: `16` sequences per forward pass
- **Sequence Length ($T$)**: `512` context tokens
- **Gradient Accumulation**: `1` (or `2` for effective batch size 32)
- **Tokens Per Step**: $16 \times 512 \times 1 = 8,192\text{ tokens/step}$
- **Total Training Steps**: `3,000` steps
- **Total Tokens Processed**: $3,000 \times 8,192 = 24,576,000\text{ tokens}$ ($\approx 24.58\text{M}$ tokens)
- **Training Corpus Size**: `509,677` tokens (audited FIFA World Cup 2022 corpus)
- **Effective Corpus Passes (Epochs)**: $\frac{24,576,000}{509,677} \approx 48.22\text{ passes (epochs)}$

### Why ~48 Epochs is Mathematically Sound for TouchlineLM:
1. **Domain-Specific Density**: Unlike general web crawl corpora where single-pass training is standard, TouchlineLM is a compact 4.2M-parameter domain specialist learning dense tabular, statistical, and tactical distributions.
2. **Regularization Balance**: With decoupled weight decay (`0.1`) and dropout (`0.1`), multi-epoch training over structured football narratives enables deep syntactic and factual pattern recognition without catastrophic overfitting.
3. **Wall-Clock Feasibility**: On a standard Google Colab NVIDIA T4 GPU (16 GB VRAM), 3,000 steps take approximately **25–35 minutes** to train to full convergence.

> 🛑 **MANUAL ACTIVATION REQUIRED**: The training command execution cell below is **commented out by default** (`# !{full_cmd_str}`). It will never run automatically. When you are ready to begin full pre-training, follow the instructions in the cell below.

In [ ]:
# ==============================================================================
# FULL PRE-TRAINING HYPERPARAMETERS
# Edit any configuration values below prior to launching full training.
# ==============================================================================

FULL_TRAINING_CONFIG = {
    # Compute & Batch Dimensions
    "device": "cuda",             # Must be 'cuda' for GPU training
    "batch_size": 16,             # Number of sequences per forward pass
    "seq_len": 512,               # Context length per sequence (<= 1024)
    "grad_accum_steps": 1,        # Number of micro-batches to accumulate gradients

    # Optimization Dynamics (AdamW)
    "learning_rate": 6e-4,        # Peak learning rate after warmup
    "min_lr": 6e-5,               # Minimum learning rate floor (10% of peak)
    "weight_decay": 0.1,          # Decoupled weight decay for 2D weight matrices
    "grad_clip": 1.0,             # Gradient norm clipping

    # Learning Schedule
    "max_steps": 3000,            # Total training steps
    "warmup_steps": 200,          # Linear warmup steps

    # Evaluation & Persistence
    "eval_interval": 250,         # Run validation evaluation every N steps
    "eval_iters": 40,             # Number of batches per validation evaluation
    "save_interval": 500,         # Save persistent checkpoint every N steps
    "log_interval": 10,           # Log training metrics every N steps
    "checkpoint_dir": str(GDRIVE_CHECKPOINT_DIR), # Google Drive persistent storage
    "seed": 42,                   # Deterministic seed
}

tokens_per_step = (
    FULL_TRAINING_CONFIG["batch_size"]
    * FULL_TRAINING_CONFIG["seq_len"]
    * FULL_TRAINING_CONFIG["grad_accum_steps"]
)
total_tokens_seen = tokens_per_step * FULL_TRAINING_CONFIG["max_steps"]
epochs_over_corpus = total_tokens_seen / len(train_tokens)

print("=" * 70)
print("TOUCHLINELM: FULL PRE-TRAINING SPECIFICATION")
print("=" * 70)
for param, val in FULL_TRAINING_CONFIG.items():
    print(f"  {param:<22}: {val}")
print("-" * 70)
print(f"Tokens Processed Per Step:  {tokens_per_step:,} tokens")
print(f"Total Tokens Seen in Run:   {total_tokens_seen:,} tokens (~{total_tokens_seen/1e6:.2f}M tokens)")
print(f"Effective Corpus Epochs:    ~{epochs_over_corpus:.2f} passes over train split")
print("=" * 70)

In [ ]:
# ==============================================================================
# LAUNCH FULL PRE-TRAINING (Execute manually when ready)
# ==============================================================================
# Instructions:
# 1. Review FULL_TRAINING_CONFIG in the previous cell.
# 2. To initiate training, uncomment the last line '!{full_cmd_str}' below.
# 3. Press Shift+Enter or click Run. Training will execute scripts/train.py.
# ==============================================================================

# Build command line invocation from FULL_TRAINING_CONFIG
train_cli_cmd = [
    "python", "scripts/train.py",
    "--device", FULL_TRAINING_CONFIG["device"],
    "--batch-size", str(FULL_TRAINING_CONFIG["batch_size"]),
    "--seq-len", str(FULL_TRAINING_CONFIG["seq_len"]),
    "--grad-accum-steps", str(FULL_TRAINING_CONFIG["grad_accum_steps"]),
    "--learning-rate", str(FULL_TRAINING_CONFIG["learning_rate"]),
    "--min-lr", str(FULL_TRAINING_CONFIG["min_lr"]),
    "--weight-decay", str(FULL_TRAINING_CONFIG["weight_decay"]),
    "--grad-clip", str(FULL_TRAINING_CONFIG["grad_clip"]),
    "--max-steps", str(FULL_TRAINING_CONFIG["max_steps"]),
    "--warmup-steps", str(FULL_TRAINING_CONFIG["warmup_steps"]),
    "--eval-interval", str(FULL_TRAINING_CONFIG["eval_interval"]),
    "--eval-iters", str(FULL_TRAINING_CONFIG["eval_iters"]),
    "--save-interval", str(FULL_TRAINING_CONFIG["save_interval"]),
    "--log-interval", str(FULL_TRAINING_CONFIG["log_interval"]),
    "--checkpoint-dir", FULL_TRAINING_CONFIG["checkpoint_dir"],
    "--seed", str(FULL_TRAINING_CONFIG["seed"]),
]

full_cmd_str = " ".join(train_cli_cmd)
print("Configured CLI Command:")
print(full_cmd_str)
print("\n" + "=" * 70)
print("To start training, uncomment the line below and re-run this cell:")
print("# !{full_cmd_str}")
print("=" * 70)

# !{full_cmd_str}

## 5. Resumption & Interactive Inference

### How to Resume After Disconnection:
Google Colab runtimes occasionally disconnect due to inactivity or session duration limits. Because all checkpoints are written atomically to **Google Drive** (`/content/drive/MyDrive/TouchlineLM_Checkpoints`), you will never lose progress.

To resume:
1. Re-run **Section 1** to mount Google Drive and install requirements.
2. Run the resumption cell below pointing to `best_model.pt` or `checkpoint_step_XXXX.pt`.
3. The training engine restores model weights, optimizer momentum, cosine decay learning rate, and RNG states without restarting warmup.

In [ ]:
# ==============================================================================
# RESUME TRAINING FROM CHECKPOINT (Run if interrupted)
# ==============================================================================

LATEST_CHECKPOINT = GDRIVE_CHECKPOINT_DIR / "best_model.pt"

if LATEST_CHECKPOINT.exists():
    print(f"Resuming training from checkpoint: {LATEST_CHECKPOINT}")
    resume_cmd = (
        f"python scripts/train.py "
        f"--device cuda "
        f"--resume '{LATEST_CHECKPOINT}' "
        f"--checkpoint-dir '{GDRIVE_CHECKPOINT_DIR}' "
        f"--batch-size {FULL_TRAINING_CONFIG['batch_size']} "
        f"--seq-len {FULL_TRAINING_CONFIG['seq_len']} "
        f"--grad-accum-steps {FULL_TRAINING_CONFIG['grad_accum_steps']} "
        f"--max-steps {FULL_TRAINING_CONFIG['max_steps']}"
    )
    print(f"Command: {resume_cmd}\n")
    # Uncomment the line below to execute resumption:
    # !{resume_cmd}
else:
    print(f"No checkpoint found at {LATEST_CHECKPOINT}. Check your Google Drive path.")

In [ ]:
# ==============================================================================
# INTERACTIVE TACTICAL TEXT GENERATION
# Sample domain-adapted narratives, player dossiers, and tactical analytics
# ==============================================================================

import torch
from src.model.model import TouchlineNanoGPT

EVAL_CKPT = GDRIVE_CHECKPOINT_DIR / "best_model.pt"

if EVAL_CKPT.exists():
    eval_model = TouchlineNanoGPT(model_config).to("cuda")
    ckpt_payload = torch.load(EVAL_CKPT, map_location="cuda", weights_only=False)
    eval_model.load_state_dict(ckpt_payload["model_state_dict"])
    eval_model.eval()

    step_restored = ckpt_payload.get("step", "Unknown")
    val_loss = ckpt_payload.get("val_loss", "N/A")
    val_ppl = ckpt_payload.get("val_perplexity", "N/A")
    print("=" * 65)
    print(f"LOADED TRAINED CHECKPOINT (Step: {step_restored})")
    print(f"Validation Loss: {val_loss} | Validation Perplexity: {val_ppl}")
    print("=" * 65)

    test_prompts = [
        "# Match Report: Argentina vs France\n**Competition Stage**: Final | **Date**: 2022-12-18\n**Final Scoreline**: Argentina",
        "# Analytical Player Dossier: Lionel Andrés Messi Cuccittini\n**Position**: Right Wing\n\n## 1. Distribution and Ball Progression",
        "# Match Report: Croatia vs Brazil\n**Competition Stage**: Quarter-finals",
        "## 1. Distribution and Ball Progression\nOperating in possession, the player executed",
    ]

    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n--- PROMPT {i} ---")
        print(prompt)
        prompt_tokens = tokenizer.encode(prompt, allowed_special="all")
        input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device="cuda")

        with torch.no_grad():
            generated_ids = eval_model.generate(
                input_ids,
                max_new_tokens=100,
                temperature=0.75,
                top_k=30
            )

        completed_text = tokenizer.decode(generated_ids[0].tolist())
        print(f"\n--- MODEL GENERATION {i} ---")
        print(completed_text)
        print("-" * 65)
else:
    print(f"No checkpoint available at {EVAL_CKPT}. Execute full training first.")